# Lot 44 — Router OOS validation
Validation hors échantillon, test verrouillé, sans entraînement SRN ni labels Minimax.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL='https://github.com/GlennEriss/songo-fish.git'
REPO_DIR=Path('/content/songo-fish')
if not REPO_DIR.exists(): subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True)
else: subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train,perf,dev]'],check=True)
print('Repository:',Path.cwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/songo-ai')
LOT41=ROOT/'experiments/lot41_deep_mcts_convergence'
LOT43=ROOT/'experiments/lot43_multi_fidelity_teacher_protocol'
OUTPUT=ROOT/'experiments/lot44_router_out_of_sample_validation'
POSITIONS=ROOT/'datasets/lot44_oos_positions.json'
EXPORT=ROOT/'exports/lot44_results.tar.gz'
OUTPUT.mkdir(parents=True,exist_ok=True); EXPORT.parent.mkdir(parents=True,exist_ok=True)
RUN=[sys.executable,'apps/trainer/scripts/run_lot44_router_oos_validation.py']
BASE=['--lot41',str(LOT41),'--lot43',str(LOT43),'--output',str(OUTPUT)]

## Préparation immuable du corpus et des partitions

In [ ]:
subprocess.run(RUN+['--stage','validate']+BASE,check=True)
subprocess.run(RUN+['--stage','corpus','--positions-file',str(POSITIONS)]+BASE,check=True)
subprocess.run(RUN+['--stage','split']+BASE,check=True)

## Recherches reprenables
Chaque combinaison partition/budget peut être relancée après déconnexion; les shards terminés sont conservés.

In [ ]:
for partition in ('train','calibration','test'):
    for budget in (4096,8192,32768,65536):
        print('SEARCH',partition,budget,flush=True)
        subprocess.run(RUN+['--stage','search','--partition',partition,'--budget',str(budget),'--device','auto','--concurrency','8']+BASE,check=True)

## Développement sans lecture des labels TEST

In [ ]:
for stage in ('features','train','calibrate','freeze'):
    subprocess.run(RUN+['--stage',stage]+BASE,check=True)

## Déverrouillage unique du TEST et arrêt contrôlé
Ne pas réexécuter pour retuner après observation des résultats.

In [ ]:
subprocess.run(RUN+['--stage','evaluate']+BASE,check=True)
subprocess.run(RUN+['--stage','finalize']+BASE,check=True)
subprocess.run(RUN+['--stage','export','--bundle',str(EXPORT)]+BASE,check=True)
print((OUTPUT/'decision.json').read_text())
print('Export:',EXPORT)